In [ ]:
# =============================================================
# Chapter 34: Teaching Architecture Without Teaching Syntax
# Pedagogical Frameworks for Agentic Systems
# Book: Design of Agentic Systems with Case Studies
# INFO 7375: Prompt Engineering for Generative AI
# =============================================================

# Chapter 34: Teaching Architecture Without Teaching Syntax
## Pedagogical Frameworks for Agentic Systems

**Book:** Design of Agentic Systems with Case Studies
**Architectural Claim:** The primary failure mode in organizational AI education is
teaching features rather than principles — producing teams that can use today's
framework but cannot evaluate tomorrow's.

### Learning Outcomes
By the end of this notebook, you will be able to:
1. Identify the gap between procedural fluency and architectural judgment
2. Apply the ICE framework as an instructional instrument (not just a prioritization tool)
3. Trigger the syntax-trap failure mode deliberately and diagnose its cause
4. Design a human decision node into an AI-assisted learning activity

## Setup

In [8]:
import openai
import json
import textwrap

client = openai.OpenAI(api_key="key")  # replace with your key

def call_llm(system_prompt, user_prompt, model="gpt-4o"):
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ]
    )
    return response.choices[0].message.content

---
## Part 1: The Demo-Capability Gap

The opening scenario of Chapter 34: a cohort completes 8 weeks of training,
demos a functioning agentic pipeline, receives applause. Three months later
they cannot evaluate a vendor's architecture.

This section demonstrates WHY. We will build a "working demo" agent and show
exactly what it does and does not certify.

In [9]:
# A minimal agentic pipeline — the kind a syntax-first cohort builds for their capstone.
# It runs. It produces output. It looks like understanding.

SYSTEM_SYNTAX = """
You are a customer service agent. You have three subagents:
- billing_agent: handles billing inquiries
- technical_agent: handles technical support
- account_agent: handles account changes

Route the customer message to the correct subagent and return their response.
"""

def syntax_first_agent(customer_message):
    """
    A syntax-fluent implementation. The student knows how to call this.
    They do not necessarily know WHY the routing works — or when it will fail.
    """
    routing_response = call_llm(
        SYSTEM_SYNTAX,
        f"Customer message: {customer_message}\nWhich subagent handles this? Reply with ONLY the subagent name."
    )
    routed_to = routing_response.strip()
    final_response = call_llm(
        f"You are the {routed_to}. Handle the customer's request professionally.",
        customer_message
    )
    return {"routed_to": routed_to, "response": final_response}

# Run the demo — this is the capstone moment
result = syntax_first_agent("I have a billing error that requires a technical fix on my account.")
print("=== CAPSTONE DEMO OUTPUT ===")
print(f"Routed to: {result['routed_to']}")
print(f"Response: {result['response'][:300]}...")
print("\n✓ Demo runs. Stakeholders applaud.")

=== CAPSTONE DEMO OUTPUT ===
Routed to: billing_agent
Response: I'm sorry to hear that you're experiencing a billing error. I'll do my best to help resolve this for you. Could you please provide more details about the error you're encountering? Any specific information, such as the error message, incorrect charge, or relevant dates, would be helpful in addressin...

✓ Demo runs. Stakeholders applaud.


### What the demo certified:
- The student can call an LLM with a system prompt ✓
- The student can chain two LLM calls ✓
- The output is coherent ✓

### What the demo did NOT test:
- What happens when the customer's issue spans multiple subagents?
- Who decides when control transfers between subagents?
- Is the orchestrator reasoning or routing? (These are architecturally different)
- What is the failure mode when the routing decision is wrong?

The demo cannot answer these questions. It was not designed to.

---
## Part 2: The AI Scaffold — ICE Scoring Generator

This is the bounded AI task: given an architectural decision, the scaffold
generates a candidate ICE score. This reduces the cognitive load of
option generation — freeing the student to focus on EVALUATION.

The scaffold halts before the architectural structure is finalized.
That halt is the Human Decision Node.

In [10]:
ICE_SCAFFOLD_SYSTEM = """
You are an architectural analysis assistant. Given an architectural decision
in an agentic system, generate a candidate ICE score (Impact, Confidence, Effort)
on a scale of 1-10 for each dimension.

For each score, provide TWO interpretations:
1. Syntax-level interpretation (surface reasoning)
2. Architectural-level interpretation (causal chain reasoning)

Return ONLY valid JSON in this exact format:
{
  "decision": "string",
  "impact": {
    "score": number,
    "syntax_interpretation": "string",
    "architectural_interpretation": "string"
  },
  "confidence": {
    "score": number,
    "syntax_interpretation": "string",
    "architectural_interpretation": "string"
  },
  "effort": {
    "score": number,
    "syntax_interpretation": "string",
    "architectural_interpretation": "string"
  },
  "architectural_assumption": "string",
  "proposed_human_decision": "string"
}
"""

def generate_ice_scaffold(architectural_decision: str) -> dict:
    """
    AI Scaffold: generates candidate ICE scores for a given architectural decision.
    This is the bounded enumeration task — the AI does the option generation,
    the human does the evaluation.
    """
    raw = call_llm(ICE_SCAFFOLD_SYSTEM, f"Architectural decision: {architectural_decision}")
    # strip markdown fences if present
    cleaned = raw.strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip()
    return json.loads(cleaned)

# The architectural decision we are scoring
decision = "Place the human-in-the-loop checkpoint AFTER the orchestrator selects a subagent, but BEFORE the subagent executes its tool calls."

print("=== AI SCAFFOLD: Generating candidate ICE scores ===")
print(f"Decision under evaluation:\n{decision}\n")

ice = generate_ice_scaffold(decision)

print(f"IMPACT: {ice['impact']['score']}/10")
print(f"  Syntax-level:       {ice['impact']['syntax_interpretation']}")
print(f"  Architectural:      {ice['impact']['architectural_interpretation']}")
print()
print(f"CONFIDENCE: {ice['confidence']['score']}/10")
print(f"  Syntax-level:       {ice['confidence']['syntax_interpretation']}")
print(f"  Architectural:      {ice['confidence']['architectural_interpretation']}")
print()
print(f"EFFORT: {ice['effort']['score']}/10")
print(f"  Syntax-level:       {ice['effort']['syntax_interpretation']}")
print(f"  Architectural:      {ice['effort']['architectural_interpretation']}")
print()
print(f"Embedded assumption: {ice['architectural_assumption']}")
print(f"Proposed human decision: {ice['proposed_human_decision']}")

=== AI SCAFFOLD: Generating candidate ICE scores ===
Decision under evaluation:
Place the human-in-the-loop checkpoint AFTER the orchestrator selects a subagent, but BEFORE the subagent executes its tool calls.

IMPACT: 8/10
  Syntax-level:       This decision places a checkpoint that ensures human oversight at a critical juncture, likely minimizing erroneous agent actions.
  Architectural:      By placing the checkpoint after subagent selection but before tool execution, the system provides a safety mechanism that can reduce errors related to inappropriate subagent-tool matches while still keeping operational flow efficient.

CONFIDENCE: 7/10
  Syntax-level:       There's a moderately high likelihood that this decision will effectively balance automation and human oversight.
  Architectural:      Given the system's design, incorporating human checkpoints at this juncture is strategically placed thereby increasing oversight confidence, though it may depend on the effectiveness of both 

---
## ⚠️ MANDATORY HUMAN DECISION NODE

The scaffold above has proposed an architectural framing and ICE scores.
**The scaffold does not know your operational context.**

Before proceeding to Part 3, you must answer the following in writing:

In [12]:
# MANDATORY HUMAN DECISION NODE
# ============================================================
# The ICE scaffold above assumes:
#   [X] The checkpoint placement decision is the highest-leverage
#       architectural decision in this pipeline.
#
# Before proceeding: verify this assumption holds for your use case.
#
# Required: Answer these three questions before running Part 3.
# ============================================================

print("\n--- Human Decision Node Input ---")
accept_impact = input("Do you accept the scaffold's impact score? (True/False): ").lower() == 'true'
justification = input("Your justification (must reference causal chain, not surface appearance): ")
assumption = input("What assumption did you reject or confirm?: ")

human_decision = {
    "do_you_accept_the_scaffolds_impact_score": accept_impact,
    "your_justification": justification,
    "what_assumption_did_you_reject_or_confirm": assumption,
    "your_corrected_ice_score_if_different": None  # dict or None
}

# FILL THIS IN BEFORE RUNNING PART 3
# Example:
# human_decision["do_you_accept_the_scaffolds_impact_score"] = False
# human_decision["your_justification"] = (
#     "The scaffold scored Impact=8 because the checkpoint intercepts tool execution. "
#     "I reject this: in my deployment the orchestrator's subagent selection is deterministic "
#     "given the routing rules, so intercepting before tool execution gives no additional "
#     "oversight over the consequential decision, which was made upstream at routing. "
#     "The checkpoint should be placed at the routing decision, not after it. Impact = 4."
# )
# human_decision["what_assumption_did_you_reject_or_confirm"] = (
#     "Rejected: that subagent selection is a reasoning step requiring oversight. "
#     "In this system it is a lookup, not a judgment."
# )

# ============================================================
# HARD STOP — do not run Part 3 until human_decision is filled in
# ============================================================
def check_human_decision(hd):
    required = [
        "do_you_accept_the_scaffolds_impact_score",
        "your_justification",
        "what_assumption_did_you_reject_or_confirm"
    ]
    missing = [k for k in required if hd.get(k) is None]
    if missing:
        raise RuntimeError(
            f"\n{'='*60}\n"
            f"HUMAN DECISION NODE NOT COMPLETE\n"
            f"You must fill in: {missing}\n"
            f"The scaffold cannot finalize the architectural structure.\n"
            f"Document your verification or rejection above before proceeding.\n"
            f"{'='*60}"
        )
    print("\n✓ Human Decision Node complete. Proceeding to Part 3.")
    print(f"  Decision accepted: {hd['do_you_accept_the_scaffolds_impact_score']}")
    print(f"  Justification: {hd['your_justification'][:150]}...")

check_human_decision(human_decision)


--- Human Decision Node Input ---
Do you accept the scaffold's impact score? (True/False): False
Your justification (must reference causal chain, not surface appearance): Proposed impact 8 because the checkpoint intercepts tool execution. I reject this. In this system, the orchestrator's sub-agent selection is a classification decision, not a reasoning step. The consequential decision was already made upstream at routing. The checkpoint is downstream of the point of no return. The impact therefore should be around 3 and not 8. 
What assumption did you reject or confirm?: The assumption that AI proposed impact 8, the assumption I rejected, was that the AI cannot propose 8. In this case, the reason was that orchestrator sub-agent selection is a classification decision and not a reasoning step, and the consequential decision was already made upstream at routing. Therefore, the checkpoint is downstream of the point of no return, and impact should be around 3. 

✓ Human Decision Node compl

---
## Part 3: The Deliberate Failure Case

**This cell is designed to fail. Run it. Observe what breaks and why.**

The failure demonstrates the syntax trap: a system that passes every
syntax-level check but fails the architectural one.

In [13]:
# DELIBERATE FAILURE CASE
# ============================================================
# The syntax-first agent from Part 1 routes correctly under normal conditions.
# Here we trigger the architectural failure mode:
# a message that spans ALL three subagent domains simultaneously.
#
# A syntax-level student predicts: "it will route to one agent, maybe wrong"
# An architecturally-trained student predicts: "the routing logic has no
# mechanism for multi-domain queries — there is no orchestrator reasoning
# here, only a classifier. It will pick one domain and silently drop the others."
# ============================================================

ADVERSARIAL_MESSAGE = (
    "My account was charged incorrectly for a technical support call "
    "that I never requested, and I need the charge reversed, the technical "
    "issue that caused the erroneous charge investigated, and my account "
    "settings updated so this cannot happen again."
)

print("=== DELIBERATE FAILURE CASE ===")
print(f"Input: {ADVERSARIAL_MESSAGE}\n")

result_failure = syntax_first_agent(ADVERSARIAL_MESSAGE)

print(f"Routed to: {result_failure['routed_to']}")
print(f"Response preview: {result_failure['response'][:400]}...")
print()
print("=== FAILURE DIAGNOSIS ===")
print("""
What broke:
  The routing component classified the message into ONE domain.
  The other two domains were silently dropped.
  No error was raised. The system returned a coherent response.
  The capstone demo would have PASSED this output.

Why the architecture caused the failure (not the model):
  The system has no orchestrator — it has a classifier dressed as an orchestrator.
  A true orchestrator reasons about which components to invoke and in what sequence.
  This system asks "which bucket does this belong in?" — a routing question.
  Multi-domain queries are not a bucket. The architecture has no mechanism for them.

What a syntax-first student sees:
  "The model got confused by the complex message."

What an architecturally-trained student sees:
  "The routing component is not an orchestrator. It is a single-label classifier.
   The failure mode was predictable from the architecture before the message was sent."

The falsification condition (from Activity 34.1):
  This component fails when the customer's issue requires reasoning across
  domain boundaries — i.e., when the correct answer is not in any single bucket.
  That condition was derivable from the specification before the demo ran.
""")

=== DELIBERATE FAILURE CASE ===
Input: My account was charged incorrectly for a technical support call that I never requested, and I need the charge reversed, the technical issue that caused the erroneous charge investigated, and my account settings updated so this cannot happen again.

Routed to: billing_agent
Response preview: I'm sorry to hear about the inconvenience you've experienced. Let's work through this together to resolve the issue. 

Firstly, I'll need to verify your account information to locate your records. Could you please provide your full name, account number, and any other relevant details you have regarding this charge?

Once I have that information, I'll review your account to confirm the charge and b...

=== FAILURE DIAGNOSIS ===

What broke:
  The routing component classified the message into ONE domain.
  The other two domains were silently dropped.
  No error was raised. The system returned a coherent response.
  The capstone demo would have PASSED this output.

---
## Part 4: The Try Exercise

**Your turn. Modify the system to trigger a different failure mode.**

Current failure: single-label routing on multi-domain query.

Challenge: Modify `syntax_first_agent` so that it attempts to handle
multi-domain queries by calling ALL THREE subagents and merging responses.
Predict the new failure mode BEFORE you run it.
Write your falsification condition here:

In [ ]:
# YOUR FALSIFICATION CONDITION (write before running):
# ============================================================
your_prediction = """
REPLACE THIS with your prediction:
- What will break when all three subagents are called?
- What is the specific architectural assumption that will fail?
- What condition triggers it?
"""
print("Your prediction:")
print(your_prediction)

In [ ]:
# Now implement the multi-subagent version and run it.
# Compare the actual failure to your prediction.
# The gap between prediction and outcome is your calibration data.

def multi_agent_attempt(customer_message):
    """
    Modified version — calls all three subagents.
    What breaks? Predict before running.
    """
    subagents = ["billing_agent", "technical_agent", "account_agent"]
    responses = {}
    for agent in subagents:
        responses[agent] = call_llm(
            f"You are the {agent}. Handle only the part of the request relevant to your domain. "
            f"If nothing in the request is relevant to your domain, say 'NOT APPLICABLE'.",
            customer_message
        )
    # Merge — no orchestrator reasoning here either
    merged = call_llm(
        "You are a response synthesizer. Merge these three agent responses into one coherent reply. "
        "Remove redundancy. Resolve contradictions by picking the most confident statement.",
        json.dumps(responses)
    )
    return {"individual_responses": responses, "merged": merged}

print("=== TRY EXERCISE: Multi-agent attempt ===")
result_try = multi_agent_attempt(ADVERSARIAL_MESSAGE)
print("Individual responses:")
for agent, resp in result_try["individual_responses"].items():
    print(f"  {agent}: {resp[:150]}...")
print()
print(f"Merged response: {result_try['merged'][:400]}...")
print()
print("Now compare to your prediction above.")
print("What did the synthesizer do when responses contradicted each other?")
print("Was there an orchestrator reasoning about the merge — or just a classifier?")

---
## Summary: What This Notebook Demonstrates

| Element | Location | What it shows |
|---|---|---|
| AI Scaffold | Part 2 | Bounded task: ICE score generation. Reduces cognitive load of option generation. |
| Human Decision Node | Part 2 hard stop | Student must evaluate scaffold output before proceeding. Cannot be skipped. |
| Deliberate failure case | Part 3 | Syntax-first agent fails on multi-domain query. Failure is architectural, not syntactic. |
| Try exercise | Part 4 | Reader triggers a new failure mode. Prediction vs. outcome = calibration data. |

**The book's master claim, demonstrated:**
Architecture is the leverage point. The model routed correctly in Part 1.
The model failed in Part 3. The model did not change. The architecture did.
The failure was predictable from the architecture before the demo ran.
The capstone demo would not have predicted it.